# 02 · Train / Validation split by time

The train folder covers 6 days (Nov 9–14). We hold out the **last day** for validation:

- **Train:** Nov 9–13. The model learns from these days.
- **Validation:** Nov 14. We check the model here while building it.
- **Test:** the locked `MINDsmall_test` folder (Nov 15). Only for final results.



In [3]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.data import BUCKETS, explode_impressions, load_behaviors, load_news, split_by_time

In [4]:
news = load_news()
behaviors = load_behaviors()
train, val = split_by_time(behaviors)

pd.DataFrame({
    "impressions": [len(train), len(val)],
    "share_%": [round(100 * len(train) / len(behaviors), 1), round(100 * len(val) / len(behaviors), 1)],
    "users": [train.user_id.nunique(), val.user_id.nunique()],
    "first": [train.time.min(), val.time.min()],
    "last": [train.time.max(), val.time.max()],
}, index=["train", "validation"])

,impressions,share_%,users,first,last
train,126695,80.7,46012,2019-11-09 00:00:19,2019-11-13 23:59:21
validation,30270,19.3,20179,2019-11-14 00:00:00,2019-11-14 23:59:13


## Check 1: Does the click history leak the future?

Each impression comes with the user's `history` (articles they clicked before). If that history included clicks from Nov 14, training would secretly see validation behavior.

In MIND, history is collected **before** the 6-day log window, so it should be identical across all of a user's impressions. Let's confirm.

In [5]:
users_with_changing_history = (behaviors.groupby("user_id").history.nunique() > 1).sum()
print(f"Users whose history changes between impressions: {users_with_changing_history}")

Users whose history changes between impressions: 0


## Check 2: How many validation users are new?

A "new" user never appears in Nov 9–13. If they have a click history, the user tower can still describe them from it.

In [6]:
new_users = set(val.user_id) - set(train.user_id)
new_user_rows = val[val.user_id.isin(new_users)]
print(f"New users: {len(new_users):,} of {val.user_id.nunique():,} validation users ({100 * len(new_users) / val.user_id.nunique():.0f}%)")
print(f"Their share of validation impressions: {100 * len(new_user_rows) / len(val):.0f}%")
print(f"New users with an empty history: {(new_user_rows.drop_duplicates('user_id').history == '').sum():,}")

New users: 3,988 of 20,179 validation users (20%)
Their share of validation impressions: 16%
New users with an empty history: 94


## Check 3: How many validation articles are brand new?

News is fresh every day. An article is "new" if it never appears in training, neither shown on Nov 9–13 nor in anyone's history.

In [7]:
train_shown = explode_impressions(train, news)
val_shown = explode_impressions(val, news)

seen_in_train = set(train_shown.news_id) | set(" ".join(train.history).split())
val_articles = val_shown.news_id.drop_duplicates()
is_new = ~val_shown.news_id.isin(seen_in_train)

print(f"Validation articles never seen in training: {100 * (~val_articles.isin(seen_in_train)).mean():.0f}%")
print(f"Share of validation clicks that go to those new articles: {100 * is_new[val_shown.clicked == 1].mean():.0f}%")

Validation articles never seen in training: 53%
Share of validation clicks that go to those new articles: 36%


## Check 4: Does the bucket mix look similar?

In [8]:
pd.DataFrame({
    "train_click_share_%": train_shown[train_shown.clicked == 1].bucket.value_counts(normalize=True),
    "val_click_share_%": val_shown[val_shown.clicked == 1].bucket.value_counts(normalize=True),
}).reindex(BUCKETS).mul(100).round(1)

,train_click_share_%,val_click_share_%
bucket,,
News,31.4,27.9
Learn,13.8,14.3
Lifestyle,14.9,17.4
Entertain,27.1,32.1
Sports,12.8,8.4
